# Week 1 · Notebook 01 — Real Data: Downloading Kepler & TESS Light Curves from MAST
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/01_download_and_plot.ipynb)

**Goal:** download a real light curve of a real planet-hosting star from NASA's archive (MAST),
understand what is inside the file, and plot it.

**Background reading (30 min, do it first):**
- Lightkurve tutorial: [Searching & downloading data](https://lightkurve.github.io/lightkurve/tutorials/1-getting-started/searching-for-data-products.html)
- HEASARC: [Understanding LightCurve objects](https://heasarc.gsfc.nasa.gov/docs/tess/LightCurve-object-Tutorial.html)

In [ ]:
# --- Setup: run this cell first. Works in Google Colab and locally. ---
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR fork
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})
print("ready, repo root:", ROOT)

## 1. Search MAST
`lightkurve.search_lightcurve` asks the MAST archive what data exist. Nothing downloads yet.
Kepler-10 hosts Kepler-10 b: the first rocky planet Kepler confirmed (Batalha et al. 2011).

In [ ]:
import lightkurve as lk
TARGET = "Kepler-10"          # try later: "pi Men", "TOI-270", "Kepler-8", "WASP-18"
search = lk.search_lightcurve(TARGET, mission="Kepler", author="Kepler", exptime="long")
search

Each row is one **quarter** (Kepler rotated every ~90 days) or, for TESS, one **sector** (~27 days).
`exptime` 1800 s = long cadence (30 min), 60 s = short cadence.

In [ ]:
lc_file = search[2].download()    # one quarter (Q3)
lc_file

## 2. What is inside? SAP vs PDCSAP flux
- **SAP_FLUX** — Simple Aperture Photometry: add up the pixels. Contains spacecraft systematics.
- **PDCSAP_FLUX** — after the Kepler/TESS team's Presearch Data Conditioning removed most
  instrumental trends. **Use PDCSAP for transit searches.**
- **QUALITY** — bit flags marking bad cadences (thruster firings, cosmic rays…). Lightkurve drops
  the worst ones by default.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
lc_file.select_flux("sap_flux").normalize().plot(ax=ax[0], label="SAP")
lc_file.select_flux("pdcsap_flux").normalize().plot(ax=ax[1], label="PDCSAP", color="C1")
plt.tight_layout(); plt.show()

## 3. Into our own pipeline's format
Our package `transitpipe` stores a light curve as three plain arrays. Seeing them as raw numbers
removes the magic.

In [ ]:
from transitpipe import io, detrend
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
print(len(lc), "points")
print("time  :", lc.time[:5], "(BKJD = BJD − 2454833 days)")
print("flux  :", lc.flux[:5])
print("errors:", lc.flux_err[:5])
print(f"point-to-point scatter ≈ {detrend.estimate_noise_ppm(lc):.0f} ppm (before detrending)")

In [ ]:
plt.plot(lc.time, lc.flux, ".", ms=1, color="k")
plt.xlabel("Time [BKJD days]"); plt.ylabel("Normalised PDCSAP flux")
plt.title("Kepler-10, quarters 3–4"); plt.show()

Kepler-10 b's transit is ~150 ppm deep and the noise per point is ~60 ppm+. You will not see
it by eye. Next week, detrending + BLS will find it.

## 4. Look up the star
The planet radius depends on the **star's** radius. Look these up now and record them in your
repo's README:
- [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) → search *Kepler-10* →
  stellar radius, mass, and the published planet radius (your Week 3 answer key!)
- For TESS stars: [ExoFOP-TESS](https://exofop.ipac.caltech.edu/tess/) (TIC catalogue values)

In [ ]:
STAR = {"name": "Kepler-10", "rstar": 1.065, "rstar_err": 0.009,   # R_sun (Fogtmann-Schulz+2014)
        "mstar": 0.910, "mstar_err": 0.021}                         # M_sun
STAR

### ✏️ Your turn — pick YOUR target
Everyone in the club does Kepler-10 first (shared answer key). Then each person picks their **own**
star so every repo is unique. Good choices: a confirmed planet with depth > 500 ppm and period < 10 d.
Browse the [Exoplanet Archive table](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=PSCompPars)
filtered by `disc_facility` = Kepler or TESS. See the target list in the project guide (Section 7).

1. Search, download, and plot your target (both SAP and PDCSAP).
2. Record star radius/mass + published planet period/radius in your README.
3. Bonus: does your star show variability (spots/rotation)? Estimate the period by eye.

### ✅ Week 1 Checkpoint
- [ ] Your own GitHub repo with this notebook committed and a README stating your target
- [ ] A labelled raw light-curve plot of your target

In [ ]:
# YOUR CODE HERE
MY_TARGET = "pi Men"
# s = lk.search_lightcurve(MY_TARGET, mission="TESS", author="SPOC")
# s